# First look at a historical hindcast

Start with the solved **2023** network copied from Sophia. Run the cells from top to bottom. Nothing here solves, rebuilds or overwrites the network.

You will inspect:
- the network and solve metadata;
- bidding-zone prices, their spread over time and duration curves;
- electricity supply, demand and storage;
- flows into and out of one selected zone;
- any load shedding;
- an optional first comparison with historical prices.

Use your local PyPSA-Eur Python interpreter. Change the settings in the next cell to explore a different year or region. The other HPC jobs can keep running independently.

**Required local files**
- `results/hindcast-build-2023/network.nc`
- `data/hindcast/bus_zones_2023.csv`

The repository also supplies the benchmark configuration and helper functions. The optional historical-price CSV is only needed in the final comparison section. All times below are **UTC**. Successful optimisation alone does not establish historical accuracy.

## 1. Choose the year and zones

The defaults focus on Denmark and nearby markets. Prices will still be extracted for the entire model. Set `PROJECT_ROOT` only if this notebook is opened outside the PyPSA-Eur checkout.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
from IPython.display import display

YEAR = 2023
FOCUS_ZONE = "DK_2"
PRICE_ZONES = ["DK_1", "DK_2", "DE_LU", "SE_4", "NO_2"]
START = f"{YEAR}-01-01"
END = f"{YEAR}-01-07"  # A date without a time includes that whole day.

# Example if opened in another project:
# PROJECT_ROOT = Path("~/PycharmProjects/pypsa-eur").expanduser()
PROJECT_ROOT = None

if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next(
        (p for p in candidates if (p / "scripts/hindcast.py").is_file()), None
    )
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to your local pypsa-eur checkout.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)  # The shared benchmark configuration uses repository-relative paths.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.hindcast import (
    benchmark_config, zonal_configuration, region_buses, model_prices,
    border_flows, load_shedding, read_hourly, reference_prices, paired_metrics,
)

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.2})
pd.set_option("display.max_rows", 60)
print("Project:", PROJECT_ROOT)
print("PyPSA:", pypsa.__version__)

## 2. Load the solved network

Loading can take a little while on the laptop. A populated marginal-price table is needed; an unsolved `composed_YEAR.nc` is not a substitute for the exported solution.

In [ ]:
cfg = benchmark_config("config/hindcast/benchmark.yaml", year=YEAR)
NETWORK = PROJECT_ROOT / cfg["network"]
MAPPING = PROJECT_ROOT / cfg["bus_zones"]

for path in [NETWORK, MAPPING]:
    if not path.is_file():
        raise FileNotFoundError(f"Copy this file from Sophia first: {path}")

n = pypsa.Network(NETWORK)
if n.buses_t.marginal_price.empty:
    raise ValueError("No solved bus prices found. Load the exported solved network.")
if set(pd.DatetimeIndex(n.snapshots).year) != {YEAR}:
    raise ValueError("The network snapshots do not match YEAR.")

overview = pd.Series({
    "Year": YEAR,
    "Snapshots": len(n.snapshots),
    "Buses": len(n.buses),
    "Generators": len(n.generators),
    "Storage units": len(n.storage_units),
    "AC lines": len(n.lines),
    "Links (including DC and internal market links)": len(n.links),
    "Network size (MB)": round(NETWORK.stat().st_size / 1e6, 1),
})
display(overview.to_frame("Value"))

provenance = n.meta.get("hindcast_reproduction", {})
display(pd.Series({
    "Solver": provenance.get("solver", "not recorded"),
    "Status": provenance.get("solver_status", "not recorded"),
    "Termination": provenance.get("termination_condition", "not recorded"),
    "Solver options": provenance.get("solver_options", "not recorded"),
    "Source network SHA256": provenance.get("source_sha256", "not recorded"),
}).to_frame("Recorded solve metadata"))

## 3. Read one common price per bidding zone

This reuses the benchmark's explicit mapping and its shared-market checks. DE–LU and the Irish market keep their national asset information while sharing a common price. Prices are not averaged across disagreeing buses.

The separate model regions `ESMA` (Mallorca), `FR15` (Corsica) and `GBZE` (Shetland) are retained. They have no assigned historical-price reference and are labelled **model-only** below.

In [ ]:
cfg = zonal_configuration(n, cfg)
mapping = region_buses(n, cfg["regions"])
price_checks = {}
prices = model_prices(
    n, mapping, mode="common",
    atol=cfg.get("price_atol", 0.01),
    rtol=cfg.get("price_rtol", 1e-6),
    checks=price_checks,
)

if FOCUS_ZONE not in prices:
    raise ValueError(f"Unknown FOCUS_ZONE. Available labels: {list(prices.columns)}")
selected = [zone for zone in PRICE_ZONES if zone in prices]
if not selected:
    raise ValueError("None of PRICE_ZONES is present in the network.")
missing_selected = sorted(set(PRICE_ZONES) - set(selected))
if missing_selected:
    print("Not present in this model:", missing_selected)

price_summary = prices.describe(percentiles=[0.05, 0.5, 0.95]).T
price_summary = price_summary.rename(columns={
    "count": "Hours", "mean": "Mean", "std": "Std", "min": "Min",
    "5%": "P05", "50%": "Median", "95%": "P95", "max": "Max",
})
price_summary["Negative hours"] = prices.lt(0).sum()
price_summary["Reference"] = [
    "model-only" if cfg["regions"][zone].get("unscored_reason") else "price area"
    for zone in price_summary.index
]
display(price_summary.sort_values("Mean").round(2))
print("All price statistics are EUR/MWh; these means are unweighted hourly means.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

prices[selected].resample("MS").mean().plot(ax=axes[0], marker="o")
axes[0].set(title=f"{YEAR}: monthly mean model prices", ylabel="EUR/MWh", xlabel="UTC")

for zone in selected:
    ordered = prices[zone].dropna().sort_values(ascending=False).to_numpy()
    axes[1].plot(np.arange(1, len(ordered) + 1), ordered, label=zone)
axes[1].set(title="Model price duration curves", ylabel="EUR/MWh", xlabel="Hours, sorted")
axes[1].legend()
fig.tight_layout()
plt.show()

window = prices.loc[START:END, selected]
if window.empty:
    print("The selected date window is empty. Adjust START and END.")
else:
    ax = window.plot()
    ax.set(title=f"Hourly model prices: {START} to {END}", ylabel="EUR/MWh", xlabel="UTC")
    plt.tight_layout()
    plt.show()

## 4. Electricity supply and demand

The next chart includes **generator output and storage discharge**. Pumped storage and batteries recycle electricity, so their discharge must not be interpreted as additional primary generation. Charging is reported separately.

The plots use actual dispatch, not available renewable potential. Aggregation uses the model's snapshot weights. Load shedding is excluded from the supply mix and reported separately in section 7.

In [ ]:
# Electricity buses come from the same complete zone mapping.
electricity_buses = [bus for buses in mapping.values() for bus in buses]
shedding_generators = n.generators.index[
    n.generators.carrier.str.contains("load|shedding", case=False, na=False)
]

def electric_supply_by_carrier(buses):
    generator_ids = n.generators.index[
        n.generators.bus.isin(buses) & ~n.generators.index.isin(shedding_generators)
    ]
    generator_output = n.generators_t.p.reindex(columns=generator_ids).mul(
        n.generators.loc[generator_ids, "sign"], axis=1
    )
    generator_output = generator_output.T.groupby(
        n.generators.loc[generator_ids, "carrier"]
    ).sum(min_count=1).T

    unit_ids = n.storage_units.index[n.storage_units.bus.isin(buses)]
    storage_output = n.storage_units_t.p_dispatch.reindex(columns=unit_ids)
    storage_output = storage_output.T.groupby(
        n.storage_units.loc[unit_ids, "carrier"]
    ).sum(min_count=1).T

    result = pd.concat([generator_output, storage_output], axis=1)
    result = result.T.groupby(level=0).sum(min_count=1).T
    result.index = prices.index
    return result

load_ids = n.loads.index[n.loads.bus.isin(electricity_buses)]
load_mw = n.get_switchable_as_dense("Load", "p_set")[load_ids].mul(
    -n.loads.loc[load_ids, "sign"], axis=1
)
load_by_bus = load_mw.T.groupby(n.loads.loc[load_ids, "bus"]).sum(min_count=1).T
load_by_bus.index = prices.index
demand = pd.DataFrame({
    zone: load_by_bus.reindex(columns=buses, fill_value=0).sum(axis=1, min_count=1)
    for zone, buses in mapping.items()
})

supply = electric_supply_by_carrier(electricity_buses)
generator_weights = pd.Series(
    n.snapshot_weightings.generators.to_numpy(), index=prices.index
)
annual_supply = supply.mul(generator_weights, axis=0).sum(min_count=1).div(1e6)
annual_supply = annual_supply[annual_supply.abs() > 1e-8].sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
annual_supply.plot.bar(ax=axes[0], color="#237b86")
axes[0].set(title=f"{YEAR}: gross electricity supply", ylabel="TWh", xlabel="Carrier")

demand.sum(axis=1, min_count=1).resample("W").mean().div(1e3).plot(ax=axes[1], color="#345c80")
axes[1].set(title="European model demand: weekly means", ylabel="GW", xlabel="UTC")
fig.tight_layout()
plt.show()
display(annual_supply.round(3).to_frame("Electricity supplied (TWh)"))

storage_ids = n.storage_units.index[n.storage_units.bus.isin(electricity_buses)]
storage_energy = pd.DataFrame({
    "Discharge (TWh)": n.storage_units_t.p_dispatch.reindex(columns=storage_ids)
        .mul(generator_weights.to_numpy(), axis=0).sum(min_count=1) / 1e6,
    "Charging (TWh)": n.storage_units_t.p_store.reindex(columns=storage_ids)
        .mul(generator_weights.to_numpy(), axis=0).sum(min_count=1) / 1e6,
})
display(storage_energy.groupby(n.storage_units.loc[storage_ids, "carrier"]).sum().round(3))

## 5. A closer look at the selected zone

Local generation does not have to equal local demand: inter-zone imports/exports, storage charging and losses also enter the balance. This is a visual inspection, not a full nodal-balance audit.

In [ ]:
focus_supply = electric_supply_by_carrier(mapping[FOCUS_ZONE])
focus_window = focus_supply.loc[START:END].div(1e3)
active = focus_window.columns[focus_window.abs().sum() > 1e-8]

if focus_window.empty or len(active) == 0:
    print("No supply to plot in this date window.")
else:
    # Typical carriers here inject positive power. Plot signed output if a carrier
    # includes consumption, since a stacked positive-only chart would hide it.
    if focus_window[active].lt(-1e-8).any().any():
        ax = focus_window[active].plot()
    else:
        ax = focus_window[active].clip(lower=0).plot.area(linewidth=0)
    demand.loc[START:END, FOCUS_ZONE].div(1e3).plot(
        ax=ax, color="black", linewidth=1.5, label="Demand"
    )
    ax.set(title=f"{FOCUS_ZONE}: local supply and demand", ylabel="GW", xlabel="UTC")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    plt.tight_layout()
    plt.show()

display(demand.mul(generator_weights, axis=0).sum(min_count=1).div(1e6)
        .sort_values(ascending=False).round(3).to_frame("Annual model demand (TWh)"))

## 6. Flows across the selected zone's borders

**Positive means net export leaving FOCUS_ZONE; negative means net import.**

These are flows on the model's aggregated AC lines and DC links. They are not automatically equivalent to historical commercial exchanges or reconstructed market transfer capacities. The helper handles both branch orientations; internal copperplate links are excluded.

In [ ]:
neighbours = sorted({
    b if a == FOCUS_ZONE else a
    for a, b in cfg["borders"] if FOCUS_ZONE in (a, b)
})
if not neighbours:
    focus_flows = pd.DataFrame(index=prices.index)
    print(f"{FOCUS_ZONE} has no mapped inter-zone AC/DC connections.")
else:
    focus_flows = border_flows(
        n, mapping, [(FOCUS_ZONE, neighbour) for neighbour in neighbours]
    )
    ax = focus_flows.resample("W").mean().div(1e3).plot()
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title=f"{FOCUS_ZONE}: weekly mean border flows",
           ylabel="GW (+ export, − import)", xlabel="UTC")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    plt.tight_layout()
    plt.show()

    flow_summary = pd.DataFrame({
        "Net export (TWh)": focus_flows.mul(generator_weights, axis=0).sum(min_count=1) / 1e6,
        "Gross export (TWh)": focus_flows.clip(lower=0).mul(generator_weights, axis=0).sum(min_count=1) / 1e6,
        "Gross import (TWh)": -focus_flows.clip(upper=0).mul(generator_weights, axis=0).sum(min_count=1) / 1e6,
    })
    display(flow_summary.round(3))

## 7. Load shedding and storage levels

Load shedding is unmet demand represented by emergency supply components. The shared helper applies the generator sign, including the kW-to-MW convention used in some PyPSA files. A successful optimisation can still contain load shedding, which matters for interpreting price spikes.

The storage plot shows **state of charge**, not power output. StorageUnit energy is in MWh; the plot converts it to GWh.

In [ ]:
shedding = load_shedding(n)
shed_total = shedding.sum(axis=1)
display(pd.Series({
    "Unserved electricity (MWh)": float(shed_total.mul(generator_weights).sum()),
    "Hours above 1e-6 MW shedding": int(shed_total.gt(1e-6).sum()),
    "Maximum shedding (MW)": float(shed_total.max()),
}).to_frame("Model result"))

if shed_total.gt(1e-6).any():
    ax = shed_total.plot(color="#b54b3c")
    ax.set(title="Load shedding across the model", ylabel="MW", xlabel="UTC")
    plt.show()
    display(shedding.mul(generator_weights, axis=0).sum()
            .sort_values(ascending=False).head(15).to_frame("Unserved electricity (MWh)"))

focus_units = n.storage_units.index[n.storage_units.bus.isin(mapping[FOCUS_ZONE])]
if len(focus_units):
    soc = n.storage_units_t.state_of_charge.reindex(columns=focus_units)
    soc = soc.T.groupby(n.storage_units.loc[focus_units, "carrier"]).sum(min_count=1).T
    soc.index = prices.index
    ax = soc.resample("D").mean().div(1e3).plot()
    ax.set(title=f"{FOCUS_ZONE}: daily mean storage energy", ylabel="GWh", xlabel="UTC")
    plt.tight_layout()
    plt.show()
else:
    print(f"No StorageUnits in {FOCUS_ZONE}.")

## 8. Optional: first comparison with historical prices

This cell runs only if `data/hindcast/prices_zonal_YEAR.csv` is already available. It performs **no download** and does not fill missing observations. Model-only islands have no assigned reference.

If needed, retrieve the observation file separately from your **local terminal**:
```bash
pixi run python scripts/hindcast.py fetch-prices --years 2023
```
Use the year you selected above. The importer refuses to overwrite existing annual outputs.

The comparison below uses matched hourly observations. Both weekly curves use those same hours and require the configured coverage. An incomplete week can be omitted. The error table reports coverage so missing data are visible. For the full documented comparison across all zones, use `hindcast.py benchmark --year 2023` later.

In [ ]:
OBSERVED = PROJECT_ROOT / cfg["prices"]
if not OBSERVED.is_file():
    print(f"Historical-price comparison skipped: {OBSERVED.name} is not available.")
    print("All preceding sections work without it.")
else:
    observed = reference_prices(
        read_hourly(OBSERVED),
        read_hourly(cfg["loads"]) if cfg.get("loads") else None,
        cfg["regions"],
    )
    rows = []
    for zone in selected:
        reason = cfg["regions"][zone].get("unscored_reason")
        if reason:
            rows.append({"Zone": zone, "Status": "NOT SCORED: " + reason})
            continue
        try:
            scores, _ = paired_metrics(prices[zone], observed[zone])
        except ValueError as exc:
            if str(exc) != "No valid matched observations at requested resolution":
                raise
            rows.append({"Zone": zone, "Status": "NOT SCORED: no paired observations"})
            continue
        rows.append({
            "Zone": zone,
            "Status": "scored",
            "Matched hours": scores["n_matched_hours"],
            "Coverage (%)": 100 * scores["coverage"],
            "Bias (EUR/MWh)": scores["bias"],  # Model minus observed.
            "MAE (EUR/MWh)": scores["mae"],
            "RMSE (EUR/MWh)": scores["rmse"],
            "Correlation": scores["correlation"],
        })
    display(pd.DataFrame(rows).set_index("Zone").round(3))

    reason = cfg["regions"][FOCUS_ZONE].get("unscored_reason")
    if reason:
        print(f"{FOCUS_ZONE}: {reason}")
    else:
        try:
            _, weekly_pair = paired_metrics(
                prices[FOCUS_ZONE], observed[FOCUS_ZONE], frequency="weekly",
                min_coverage=cfg.get("min_aggregate_coverage", 1.0),
            )
        except ValueError as exc:
            if str(exc) != "No valid matched observations at requested resolution":
                raise
            print(f"No sufficiently complete paired weeks for {FOCUS_ZONE}.")
        else:
            ax = weekly_pair.rename(columns={"model": "Model", "observed": "Historical"}).plot()
            ax.set(title=f"{FOCUS_ZONE}: matched weekly mean prices",
                   ylabel="EUR/MWh", xlabel="UTC")
            plt.tight_layout()
            plt.show()

## What to inspect next

- Which zones have the largest average prices and the most extreme hours?
- When do neighbouring zone prices separate, and what do model border flows show?
- Do price spikes coincide with low renewable supply or load shedding?
- Once observations are available, is a zone systematically too expensive or too cheap?
- Repeat for another **completed** annual network by changing `YEAR`, `START` and `END`, then running from the top. No other annual network is loaded automatically.

The notebook is a first inspection tool. These plots do not establish that capacities, market coupling or historical prices have been reproduced accurately. The original solved NetCDF is never modified. The notebook was committed with empty outputs; it has not been executed against your HPC result.